# Customer Churn Prediction System
### Phase 7: Final Analysis & Write-up (Tasks 20-21)

The last phase. No new modeling here -- this is about stepping back, pulling
everything from Phases 1-6 together, and writing the conclusion that ties the
whole project into one coherent story.

## Setup: rebuilding the final tuned model (same as Phase 6)

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

df = pd.read_csv('customer_churn.csv').drop(columns=['CustomerID'])
X = df.drop(columns=['Churn'])
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train_prep = X_train.copy()
X_test_prep = X_test.copy()

gender_map = {'Male': 0, 'Female': 1}
X_train_prep['Gender'] = X_train_prep['Gender'].map(gender_map)
X_test_prep['Gender'] = X_test_prep['Gender'].map(gender_map)

categorical_cols = ['Subscription Type', 'Contract Length']
X_train_prep = pd.get_dummies(X_train_prep, columns=categorical_cols, drop_first=True)
X_test_prep = pd.get_dummies(X_test_prep, columns=categorical_cols, drop_first=True)
X_test_prep = X_test_prep.reindex(columns=X_train_prep.columns, fill_value=0)

numeric_cols = ['Age', 'Tenure', 'Usage Frequency', 'Support Calls',
                'Payment Delay', 'Total Spend', 'Last Interaction']
scaler = StandardScaler()
X_train_prep[numeric_cols] = scaler.fit_transform(X_train_prep[numeric_cols])
X_test_prep[numeric_cols] = scaler.transform(X_test_prep[numeric_cols])

model = RandomForestClassifier(
    n_estimators=200, max_depth=None, min_samples_split=2, random_state=42
)
model.fit(X_train_prep, y_train)
test_pred = model.predict(X_test_prep)
test_proba = model.predict_proba(X_test_prep)[:, 1]

print("Final tuned model ready.")

Final tuned model ready.


## Task 20: Overall Analysis

### A) Model comparison -- recap from Phase 5

In [2]:
model_comparison = pd.DataFrame({
    'Model': ['Random Forest', 'SVM', 'KNN', 'Logistic Regression'],
    'Accuracy':  [0.9981, 0.9456, 0.9170, 0.8271],
    'Precision': [0.9992, 0.9290, 0.8862, 0.8138],
    'Recall':    [0.9967, 0.9584, 0.9464, 0.8234],
    'F1-Score':  [0.9979, 0.9434, 0.9153, 0.8186],
}).set_index('Model')

model_comparison

,Accuracy,Precision,Recall,F1-Score
Model,,,,
Random Forest,0.9981,0.9992,0.9967,0.9979
SVM,0.9456,0.9290,0.9584,0.9434
KNN,0.9170,0.8862,0.9464,0.9153
Logistic Regression,0.8271,0.8138,0.8234,0.8186


**Random Forest was the clear winner** across every single metric, not just
one. This matters -- a model that wins on accuracy but loses on recall would be a
genuinely harder call. Here there was no trade-off to weigh.

### B) Which features actually drove predictions

In [3]:
importances = pd.Series(model.feature_importances_, index=X_train_prep.columns)
importances = importances.sort_values(ascending=False)
print(importances.round(4))

Payment Delay                 0.4385
Support Calls                 0.1621
Tenure                        0.1115
Usage Frequency               0.0847
Gender                        0.0707
Total Spend                   0.0461
Age                           0.0409
Contract Length_Monthly       0.0267
Last Interaction              0.0092
Contract Length_Quarterly     0.0061
Subscription Type_Premium     0.0019
Subscription Type_Standard    0.0018
dtype: float64


**Two completely independent analyses agreed:** Phase 2's correlation
analysis and this model's own learned feature importance both point to
`Payment Delay` and `Support Calls` as the dominant churn drivers, with
everything else playing a distinctly smaller role. That consistency is a
stronger claim than either finding alone.

### C) Did scaling and tuning actually help?

In [4]:
tuning_summary = pd.DataFrame({
    'Stage': ['Baseline Random Forest (default settings)',
              'Manually adjusted (200 trees, depth 20, min_leaf 2)',
              'GridSearchCV best cross-validation score',
              'GridSearchCV best model, on real test set'],
    'F1-Score': [0.9979, 0.9976, 0.9975, 0.9980],
})
tuning_summary

,Stage,F1-Score
0,Baseline Random Forest (default settings),0.9979
1,"Manually adjusted (200 trees, depth 20, min_le...",0.9976
2,GridSearchCV best cross-validation score,0.9975
3,"GridSearchCV best model, on real test set",0.9980


**Honest answer: barely, and that's a meaningful finding, not a
disappointment.** The baseline model was already essentially at its ceiling
before any tuning. This happened *because* the underlying patterns in the data
are so clean (recall the sharp thresholds from Phase 2) -- there wasn't much
room left to improve. Scaling itself mattered more for the other three models
(Logistic Regression, SVM, KNN are all scale-sensitive) than for Random Forest,
which doesn't actually need scaled inputs to begin with.

### D) Which customers were hardest to classify

Out of 12,875 test customers, the tuned model got all but a handful wrong.
Let's actually look at *who* those mistakes were, instead of just reporting the
count.

In [5]:
results = X_test.copy()
results['Actual'] = y_test.values
results['Predicted'] = test_pred
results['Churn Probability'] = test_proba.round(3)

misclassified = results[results['Actual'] != results['Predicted']]
print(f"Total misclassified: {len(misclassified)} out of {len(results)} "
      f"({len(misclassified)/len(results):.2%})")

misclassified[['Age', 'Support Calls', 'Payment Delay', 'Contract Length',
               'Actual', 'Predicted', 'Churn Probability']].sort_values('Churn Probability')

Total misclassified: 24 out of 12875 (0.19%)


,Age,Support Calls,Payment Delay,Contract Length,Actual,Predicted,Churn Probability
8024,30,1,15,Quarterly,1,0,0.335
59695,39,9,17,Quarterly,1,0,0.340
55065,24,2,27,Annual,1,0,0.345
19590,41,1,13,Monthly,1,0,0.350
7660,20,1,22,Annual,1,0,0.410
56464,31,3,29,Annual,1,0,0.415
23215,40,3,10,Monthly,1,0,0.420
51974,58,3,26,Monthly,1,0,0.440
48765,45,3,26,Annual,1,0,0.445
52628,39,0,10,Annual,1,0,0.460


In [6]:
missed_churners = misclassified[(misclassified['Actual'] == 1) & (misclassified['Predicted'] == 0)]
false_alarms = misclassified[(misclassified['Actual'] == 0) & (misclassified['Predicted'] == 1)]

print(f"Missed churners (looked safe, actually left): {len(missed_churners)}")
print(f"  Average Support Calls: {missed_churners['Support Calls'].mean():.1f}  "
      f"(dataset overall: {results['Support Calls'].mean():.1f})")
print(f"  Average Payment Delay: {missed_churners['Payment Delay'].mean():.1f}  "
      f"(dataset overall: {results['Payment Delay'].mean():.1f})")

print(f"\nFalse alarms (looked risky, actually stayed): {len(false_alarms)}")
print(f"  Average Support Calls: {false_alarms['Support Calls'].mean():.1f}")
print(f"  Average Payment Delay: {false_alarms['Payment Delay'].mean():.1f}")

Missed churners (looked safe, actually left): 19
  Average Support Calls: 2.0  (dataset overall: 5.4)
  Average Payment Delay: 17.2  (dataset overall: 17.2)

False alarms (looked risky, actually stayed): 5
  Average Support Calls: 6.8
  Average Payment Delay: 23.0


**The pattern in the mistakes tells its own story.** Every single one of the
model's ~24 errors landed with a predicted probability between roughly 33% and
60% -- exactly the "unsure" zone that Phase 6 found was otherwise almost never
used. That's not a coincidence: these are precisely the customers who don't
follow the dominant Payment-Delay/Support-Calls rule.

- The **missed churners** mostly had *low* support calls and unremarkable payment
  delay -- they left for reasons the dataset doesn't capture (price sensitivity,
  a competitor's offer, something outside these 10 columns).
- The **false alarms** mostly had elevated support calls or payment delay -- they
  looked exactly like the customers who usually churn, but stayed anyway.

**This is genuinely the most interesting finding to include in your write-up:**
the model isn't failing randomly -- it's failing exactly where the data's own
clean rule breaks down, on the customers who are the real exceptions.

## Task 21: Final Conclusion

### Project Summary

This project built a full customer churn prediction pipeline across 7 phases:

1. **Data understanding** -- 64,374 customers, 12 columns, target `Churn`
   reasonably balanced at 52.6% stayed / 47.4% churned.
2. **Cleaning & EDA** -- the dataset had zero missing values and zero duplicates.
   `Payment Delay` and `Support Calls` emerged immediately as the standout
   predictors, each showing a sharp threshold effect (churn rate jumping from
   ~10-30% to ~60-77% past a specific point) rather than a gradual trend.
3. **Data preparation** -- 80/20 stratified train-test split, categorical
   encoding, and feature scaling, done carefully to avoid any leakage from the
   test set into training.
4. **Model training** -- four different algorithms trained on identical data:
   Logistic Regression, Random Forest, SVM, and KNN.
5. **Evaluation & tuning** -- Random Forest won decisively (F1 ≈ 0.998),
   confirmed the same top features found in EDA, and further tuning only
   improved it marginally because it was already near its ceiling.
6. **The prediction system** -- a working function that takes a new customer's
   raw info and returns a prediction, probability, risk level, named
   contributing factors, and a specific recommended action.
7. **This final analysis** -- confirmed the model's few mistakes cluster exactly
   where its dominant pattern breaks down.

### Key Takeaways

- **`Payment Delay` (past 15 days) and `Support Calls` (5 or more) are, by a wide
  margin, the two strongest churn signals** in this dataset -- confirmed by
  correlation analysis, visual EDA, model feature importance, and error
  analysis, four separate ways.
- **Random Forest was the right choice for this data** -- its tree structure
  naturally captures the sharp, rule-like thresholds present here, which is
  likely why it dramatically outperformed the other three models rather than
  winning by a small margin.
- **Tuning had limited impact** because the model was already close to optimal
  on default settings -- itself a useful, honest finding rather than a gap in
  the work.

### Limitations

- **The near-perfect performance (~99.8% F1) is unusually clean for real-world
  churn data.** This most likely reflects that the dataset follows fairly clean,
  rule-based generation rather than the noisier relationships typical of real
  company data. A production system trained on messier real customer data should
  expect meaningfully lower scores than this.
- **The risk system rarely lands on "Medium"** for the same reason -- the model
  is usually very confident one way or the other, which may not reflect how
  gradual real churn risk actually is.
- **This is a single snapshot, not a time series.** The model can say a customer
  looks at-risk *right now*, but can't say when they'll actually churn or how
  their risk is trending over time.
- **Only 10 features were available.** Real churn systems often have access to
  richer signals (browsing behavior, competitor pricing, customer sentiment from
  support transcripts) that could explain the ~24 exception cases this analysis
  surfaced.

### Possible Future Improvements

- Test on a messier, real-world dataset to see how much performance genuinely
  drops, and whether feature importance rankings hold up.
- Add a cost-sensitive evaluation (weighing a missed churner as more costly than
  a false alarm) directly into model selection, rather than treating both
  mistakes equally.
- Track customers over multiple time snapshots to model *when* churn risk is
  rising, not just whether it's currently high.